# BioPred Phase 0 -- Molecule-Level Modeling Table

This notebook converts the cleaned GABA-A activity evidence artifact from Notebook 3 into a molecule-level modeling table. The cleaned evidence artifact remains activity-record level, meaning a single molecule may appear in multiple rows across endpoints, targets, assays, or documents.

The goal of this notebook is to aggregate repeated evidence into one row per molecule, create molecule-level potency summaries and candidate activity labels, preserve relevant evidence diagnostics, and save a validated modeling-table artifact for downstream molecule-level EDA, split design, and baseline modeling.

The primary aggregation policy follows the recommendation from Notebook 3: median pX is used as the main molecule-level potency summary. The leading candidate activity label is `median_px >= 6.0`, while `median_px >= 5.5` and `median_px >= 6.5` are retained as sensitivity-analysis labels.

## Objectives

1. Load the cleaned activity evidence artifact from Notebook 3.
2. Validate the handoff artifact shape, schema, molecule count, and required fields.
3. Aggregate cleaned evidence records to one row per molecule.
4. Compute molecule-level potency summaries, including mean, median, minimum, maximum, and pX range.
5. Create molecule-level candidate activity labels from median pX thresholds.
6. Preserve evidence diagnostics such as record count, endpoint count, target count, assay count, pX variability, and label-conflict indicators.
7. Validate molecule-level label prevalence and aggregation outputs.
8. Save the molecule-level modeling table artifact for downstream EDA and split-readiness analysis.


In [1]:
# list imports needed for this notebook
from pathlib import Path
import sys
import os
from dotenv import load_dotenv
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
import numpy as np

# Force notebook runtime to project root
%cd /home/ryanm/projects/BioPred

# define paths and link src.paths
SRC_DIR = Path.cwd() / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import biopred.paths as paths

load_dotenv(paths.PROJECT_ROOT / ".env", override=True)

# create the database engine
db_url = URL.create(
    drivername = os.getenv("BIOPRED_DB_DIALECT"),
    username=os.getenv("BIOPRED_DB_USER"),
    password=os.getenv("BIOPRED_DB_PASSWORD"),
    host=os.getenv("BIOPRED_DB_HOST"),
    port=int(os.getenv("BIOPRED_DB_PORT")),
    database=os.getenv("BIOPRED_DB_NAME")
)

engine = create_engine(db_url, pool_pre_ping=True)
schema = os.getenv("BIOPRED_DB_SCHEMA", "public")

/home/ryanm/projects/BioPred


In [3]:
# load our df_clean artifact we created from the last notebook
clean_activity_evidence_path = paths.PROCESSED_DATA_DIR / "chembl_gabaa_clean_activity_evidence.parquet"

activity_evidence_clean = pd.read_parquet(clean_activity_evidence_path)

In [ ]:
# rename the df and verify shape
df = activity_evidence_clean

df.shape # Expecting (4435, 27)

(4435, 27)